<a href="https://colab.research.google.com/github/jeyajeevaj17/automatic_message_cipher/blob/main/AutomaticMessageCipher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import gradio as gr
import random
import hashlib
import os

# --- Emoji Base Map ---
char_to_emoji_bases = {
    "Standard": {
        'a': '😀', 'b': '😂', 'c': '😎', 'd': '😍', 'e': '🤔',
        'f': '🙄', 'g': '😴', 'h': '😡', 'i': '🤩', 'j': '🥳',
        'k': '😇', 'l': '😜', 'm': '🤖', 'n': '👻', 'o': '💩',
        'p': '👽', 'q': '👾', 'r': '🎃', 's': '🐵', 't': '🦄',
        'u': '🐸', 'v': '🦊', 'w': '🐶', 'x': '🐱', 'y': '🦁', 'z': '🐯',
        '0': '🍎', '1': '🚀', '2': '🎈', '3': '🌟', '4': '🍕',
        '5': '🐢', '6': '🎩', '7': '⚡', '8': '🍄', '9': '🎲',
        ' ': '⬜', '.': '⚫', ',': '⚪', '!': '❗', '?': '❓', '-': '➖',
        '_': '⬛', '\'': '🟤', '"': '🟠', '@': '📧', '#': '🔢',
        '%': '📊', '&': '➕', '(': '⭕', ')': '⭕', '[': '〚', ']': '〛',
        '{': '❴', '}': '❵', ':': '⏰', ';': '💉', '+': '➕', '/': '➗',
        '\\': '🛡️', '=': '🟰', '*': '✳️', '<': '◀️', '>': '▶️',
        '~': '〰️', '^': '⬆️', '$': '💲',
    }
}

UPPERCASE_MARKER = '🔺'
ALL_CHARS = list(char_to_emoji_bases["Standard"].keys())

# --- Cipher Generation ---
def generate_cipher(theme, password=None):
    base_map = char_to_emoji_bases.get(theme, char_to_emoji_bases["Standard"])
    emojis = list(base_map.values())
    chars = ALL_CHARS[:]

    # Randomize or deterministic
    if password:
        seed = int(hashlib.sha256((password + theme).encode()).hexdigest(), 16)
    else:
        seed = random.randint(0, 1_000_000_000)

    rnd = random.Random(seed)
    shuffled_emojis = emojis[:]
    rnd.shuffle(shuffled_emojis)

    cipher_map = {c: e for c, e in zip(chars, shuffled_emojis)}
    inverse_map = {v: k for k, v in cipher_map.items()}
    return cipher_map, inverse_map

# --- Encryption / Decryption ---
def encrypt_text(text, cipher_map):
    result = []
    for ch in text:
        if ch.isupper() and ch.lower() in cipher_map:
            result.append(UPPERCASE_MARKER + cipher_map[ch.lower()])
        elif ch in cipher_map:
            result.append(cipher_map[ch])
        else:
            result.append(ch)
    return "".join(result)

def decrypt_text(text, inverse_map):
    i = 0
    result = []
    while i < len(text):
        if text[i:i+len(UPPERCASE_MARKER)] == UPPERCASE_MARKER:
            emoji = text[i+len(UPPERCASE_MARKER)]
            ch = inverse_map.get(emoji, '')
            result.append(ch.upper())
            i += len(UPPERCASE_MARKER) + 1
        else:
            ch = inverse_map.get(text[i], '')
            if ch:
                result.append(ch)
            else:
                result.append(text[i])
            i += 1
    return "".join(result)

def is_emoji_encrypted(text, inverse_map):
    for emoji in inverse_map.keys():
        if emoji in text:
            return True
    return UPPERCASE_MARKER in text

def process_text(input_text, mode, theme, password):
    if not input_text:
        return "", "No input provided."
    cipher_map, inverse_map = generate_cipher(theme, password)
    if mode == "Auto":
        mode_use = "Decrypt" if is_emoji_encrypted(input_text, inverse_map) else "Encrypt"
    else:
        mode_use = mode.capitalize()
    if mode_use == "Encrypt":
        output = encrypt_text(input_text, cipher_map)
    else:
        output = decrypt_text(input_text, inverse_map)
    return output, f"{mode_use}ion done."

# --- File Processing ---
def process_file(file, mode, theme, password):
    if file is None:
        return "", "No file uploaded.", None
    try:
        with open(file.name, "r", encoding="utf-8") as f:
            content = f.read()

        output, status = process_text(content, mode, theme, password)
        new_filename = f"processed_{os.path.basename(file.name)}"
        with open(new_filename, "w", encoding="utf-8") as f:
            f.write(output)
        return output, status, new_filename
    except Exception as e:
        return "", f"File processing error: {e}", None

# --- Cipher Display ---
def generate_cipher_text(theme, password):
    cipher_map, _ = generate_cipher(theme, password)
    lines = [f"{ch if ch != ' ' else 'SPACE'} → {cipher_map[ch]}" for ch in sorted(cipher_map.keys())]
    return "\n".join(lines)

# --- UI Construction ---
with gr.Blocks() as demo:
    gr.Markdown("# 🔐 **Emoji Cipher Encoder/Decoder (Full Version)**")

    # --- TEXT MODE TAB ---
    with gr.Tab("Text Mode"):
        with gr.Row():
            theme = gr.Dropdown(list(char_to_emoji_bases.keys()), label="Emoji Theme", value="Standard")
            password = gr.Textbox(label="Password (optional)", placeholder="Enter password for consistent cipher")
            mode = gr.Radio(["Auto", "Encrypt", "Decrypt"], label="Mode", value="Auto")

        input_text = gr.Textbox(label="Input Text", lines=6, placeholder="Enter text or emoji cipher...")
        output_text = gr.Textbox(label="Output Text", lines=6, interactive=False)
        status = gr.Textbox(label="Status", interactive=False)

        with gr.Row():
            run_btn = gr.Button("▶️ Run")
            clear_btn = gr.Button("🧹 Clear")

        run_btn.click(process_text, [input_text, mode, theme, password], [output_text, status])
        clear_btn.click(lambda: ("", "", ""), None, [input_text, output_text, status])

    # --- FILE MODE TAB ---
    with gr.Tab("File Mode"):
        gr.Markdown("### 📂 Encrypt or Decrypt Entire Files")

        with gr.Row():
            theme_file = gr.Dropdown(list(char_to_emoji_bases.keys()), label="Emoji Theme", value="Standard")
            password_file = gr.Textbox(label="Password (optional)")
            mode_file = gr.Radio(["Auto", "Encrypt", "Decrypt"], label="Mode", value="Auto")

        file_input = gr.File(label="Upload UTF-8 Text File")
        file_output = gr.Textbox(label="Processed File Preview", lines=6, interactive=False)
        file_status = gr.Textbox(label="Status", interactive=False)
        output_file = gr.File(label="Download Processed File")

        with gr.Row():
            run_file_btn = gr.Button("▶️ Run File")
            clear_file_btn = gr.Button("🧹 Clear File")

        # Run manually after upload + mode select
        run_file_btn.click(process_file, [file_input, mode_file, theme_file, password_file],
                           [file_output, file_status, output_file])
        clear_file_btn.click(lambda: ("", "", None, ""), None,
                             [file_output, file_status, output_file, file_input])

    # --- CIPHER VIEWER TAB ---
    with gr.Tab("Cipher Viewer"):
        gr.Markdown("### 🔍 View Generated Cipher Mapping")
        cipher_display = gr.Textbox(label="Generated Cipher (Char → Emoji)", lines=10, interactive=False)
        generate_btn = gr.Button("🔄 Generate Cipher")
        clear_cipher_btn = gr.Button("🧹 Clear Cipher")

        generate_btn.click(generate_cipher_text, [theme, password], [cipher_display])
        clear_cipher_btn.click(lambda: "", None, [cipher_display])

demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9730ffbd6d1e1fbd19.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!pip install -q gradio scikit-learn pandas matplotlib seaborn

In [ ]:
import pandas as pd
import random

random.seed(42)

normal_messages = [
    "Hello how are you",
    "Good morning",
    "Good evening everyone",
    "How was your day",
    "The weather is nice today",
    "See you tomorrow",
    "Have a great day",
    "Thank you for your help",
    "Can you send me the notes",
    "The meeting starts at ten",
    "I will come to college tomorrow",
    "Let's have lunch together",
    "The project presentation is tomorrow",
    "Please send the assignment",
    "Are you coming to class today",
    "The exam is next week",
    "I completed the project",
    "Let's discuss the project",
    "See you at the library",
    "Have a nice weekend"
]

personal_messages = [
    "I miss you",
    "I am thinking about you",
    "Let's meet tomorrow",
    "I want to talk to you privately",
    "Please call me later",
    "I have something to tell you",
    "Can we meet after college",
    "I will tell you when we meet",
    "This is between you and me",
    "Please keep this conversation private",
    "I want to discuss something personal",
    "Can you call me tonight",
    "I need to talk to you",
    "Let's meet somewhere quiet",
    "I have a personal matter to discuss",
    "Please don't share this message",
    "I will explain everything later",
    "Can we talk privately",
    "I trust you with this",
    "Let's discuss this in person"
]

sensitive_messages = [
    "My bank account number is 1234567890",
    "My credit card details are private",
    "My account information should not be shared",
    "Please protect my financial information",
    "My bank details are confidential",
    "Here is my account number 9876543210",
    "My card number is 456789123456",
    "Please do not share my banking information",
    "My financial details are sensitive",
    "This contains private financial information",
    "My transaction details should remain private",
    "Please protect my payment information",
    "My salary information is private",
    "Do not share my financial records",
    "My insurance information is sensitive",
    "My personal identification information is private",
    "Please keep my financial records secure",
    "This message contains sensitive information",
    "My account details must be protected",
    "Keep these financial details private"
]

confidential_messages = [
    "My OTP is 583921",
    "My password is Secret123",
    "The verification code is 847291",
    "My login password is Admin@123",
    "The authentication code is 739201",
    "My security code is 492817",
    "The PIN is 4729",
    "My account password is MyPass123",
    "The OTP code is 928374",
    "My verification number is 192837",
    "The secret key is ABC123",
    "My login credentials are private",
    "Here is my authentication password",
    "The one time password is 123456",
    "My security PIN is 8392",
    "This is my confidential access code",
    "My private key is XYZ123",
    "The verification PIN is 9274",
    "My authentication token is 839201",
    "This password must not be shared"
]

data = []

for message in normal_messages:
    data.append([message, "normal"])

for message in personal_messages:
    data.append([message, "personal"])

for message in sensitive_messages:
    data.append([message, "sensitive"])

for message in confidential_messages:
    data.append([message, "confidential"])

# Increase dataset size using slight variations
all_data = data.copy()

for _ in range(25):
    for message, label in data:
        variations = [
            message,
            message.lower(),
            message + ".",
            message + " please",
            "Please " + message.lower()
        ]

        selected = random.choice(variations)
        all_data.append([selected, label])

df = pd.DataFrame(all_data, columns=["message", "label"])

df = df.drop_duplicates().reset_index(drop=True)

print("Dataset size:", len(df))
print()
print(df["label"].value_counts())
print()
display(df.head(10))

Dataset size: 400

label
normal          100
personal        100
sensitive       100
confidential    100
Name: count, dtype: int64



,message,label
0,Hello how are you,normal
1,Good morning,normal
2,Good evening everyone,normal
3,How was your day,normal
4,The weather is nice today,normal
5,See you tomorrow,normal
6,Have a great day,normal
7,Thank you for your help,normal
8,Can you send me the notes,normal
9,The meeting starts at ten,normal


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X = df["message"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Convert text into numerical features
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    max_features=5000
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train ML model
model = LogisticRegression(
    max_iter=1000
)

model.fit(X_train_tfidf, y_train)

# Test
predictions = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, predictions)

print("AI Model Accuracy:", round(accuracy * 100, 2), "%")
print()
print(classification_report(y_test, predictions))

AI Model Accuracy: 100.0 %

              precision    recall  f1-score   support

confidential       1.00      1.00      1.00        20
      normal       1.00      1.00      1.00        20
    personal       1.00      1.00      1.00        20
   sensitive       1.00      1.00      1.00        20

    accuracy                           1.00        80
   macro avg       1.00      1.00      1.00        80
weighted avg       1.00      1.00      1.00        80



In [ ]:
def classify_message(message):

    transformed = vectorizer.transform([message])

    prediction = model.predict(transformed)[0]

    probabilities = model.predict_proba(transformed)[0]

    confidence = max(probabilities)

    return prediction, confidence


test_messages = [
    "Hello, how are you?",
    "I want to talk to you privately",
    "My bank account number is 1234567890",
    "My OTP is 583921"
]

for message in test_messages:

    category, confidence = classify_message(message)

    print("Message:", message)
    print("Category:", category)
    print("Confidence:", round(confidence * 100, 2), "%")
    print("-" * 60)

Message: Hello, how are you?
Category: normal
Confidence: 69.31 %
------------------------------------------------------------
Message: I want to talk to you privately
Category: personal
Confidence: 80.07 %
------------------------------------------------------------
Message: My bank account number is 1234567890
Category: sensitive
Confidence: 72.31 %
------------------------------------------------------------
Message: My OTP is 583921
Category: confidential
Confidence: 73.0 %
------------------------------------------------------------


In [ ]:
import random
import hashlib

char_to_emoji_bases = {
    "Standard": {
        'a': '😀', 'b': '😂', 'c': '😎', 'd': '😍', 'e': '🤔',
        'f': '🙄', 'g': '😴', 'h': '😡', 'i': '🤩', 'j': '🥳',
        'k': '😇', 'l': '😜', 'm': '🤖', 'n': '👻', 'o': '💩',
        'p': '👽', 'q': '👾', 'r': '🎃', 's': '🐵', 't': '🦄',
        'u': '🐸', 'v': '🦊', 'w': '🐶', 'x': '🐱', 'y': '🦁', 'z': '🐯',

        '0': '🍎', '1': '🚀', '2': '🎈', '3': '🌟', '4': '🍕',
        '5': '🐢', '6': '🎩', '7': '⚡', '8': '🍄', '9': '🎲',

        ' ': '⬜',
        '.': '⚫',
        ',': '⚪',
        '!': '❗',
        '?': '❓',
        '-': '➖',
        '_': '⬛',
        "'": '🟤',
        '"': '🟠',
        '@': '📧',
        '#': '🔢',
        '%': '📊',
        '&': '➕',
        '(': '⭕',
        ')': '⭕',
        '[': '〚',
        ']': '〛',
        '{': '❴',
        '}': '❵',
        ':': '⏰',
        ';': '💉',
        '+': '➕',
        '/': '➗',
        '\\': '🛡️',
        '=': '🟰',
        '*': '✳️',
        '<': '◀️',
        '>': '▶️',
        '~': '〰️',
        '^': '⬆️',
        '$': '💲'
    }
}

UPPERCASE_MARKER = '🔺'

ALL_CHARS = list(
    char_to_emoji_bases["Standard"].keys()
)


def generate_cipher(theme="Standard", password=None):

    base_map = char_to_emoji_bases[theme]

    emojis = list(base_map.values())
    chars = ALL_CHARS[:]

    if password:

        seed = int(
            hashlib.sha256(
                (password + theme).encode()
            ).hexdigest(),
            16
        )

    else:

        seed = 123456

    rnd = random.Random(seed)

    shuffled_emojis = emojis[:]

    rnd.shuffle(shuffled_emojis)

    cipher_map = {
        c: e
        for c, e in zip(chars, shuffled_emojis)
    }

    inverse_map = {
        e: c
        for c, e in cipher_map.items()
    }

    return cipher_map, inverse_map

In [ ]:
def encrypt_text(text, cipher_map):

    result = []

    for ch in text:

        if ch.isupper() and ch.lower() in cipher_map:

            result.append(
                UPPERCASE_MARKER +
                cipher_map[ch.lower()]
            )

        elif ch in cipher_map:

            result.append(
                cipher_map[ch]
            )

        else:

            result.append(ch)

    return "".join(result)


def decrypt_text(text, inverse_map):

    result = []

    i = 0

    while i < len(text):

        if text.startswith(UPPERCASE_MARKER, i):

            i += len(UPPERCASE_MARKER)

            if i < len(text):

                emoji = text[i]

                ch = inverse_map.get(
                    emoji,
                    ''
                )

                result.append(ch.upper())

                i += 1

        else:

            emoji = text[i]

            ch = inverse_map.get(
                emoji,
                ''
            )

            if ch:

                result.append(ch)

            else:

                result.append(emoji)

            i += 1

    return "".join(result)

In [ ]:
def get_encryption_policy(category):

    policies = {

        "normal": {
            "name": "Standard Emoji Cipher",
            "level": "Basic"
        },

        "personal": {
            "name": "Password-Based Emoji Cipher",
            "level": "Medium"
        },

        "sensitive": {
            "name": "Password-Protected Emoji Cipher",
            "level": "High"
        },

        "confidential": {
            "name": "Strong Password-Based Emoji Cipher",
            "level": "High"
        }
    }

    return policies.get(
        category,
        policies["normal"]
    )

In [ ]:
def ai_encrypt(message, password):

    if not message.strip():

        return (
            "",
            "No message provided.",
            "",
            ""
        )

    # AI classification
    category, confidence = classify_message(
        message
    )

    # Select encryption policy
    policy = get_encryption_policy(
        category
    )

    # Use category as part of cipher seed
    adaptive_password = (
        password if password
        else "AI-" + category
    )

    # Generate cipher
    cipher_map, _ = generate_cipher(
        "Standard",
        adaptive_password
    )

    # Encrypt
    encrypted = encrypt_text(
        message,
        cipher_map
    )

    return (
        encrypted,
        category.upper(),
        f"{confidence * 100:.2f}%",
        policy["name"]
    )

In [ ]:
def ai_decrypt(encrypted_message, category, password):

    if not encrypted_message.strip():

        return "", "No encrypted message provided."

    category = category.lower()

    adaptive_password = (
        password if password
        else "AI-" + category
    )

    _, inverse_map = generate_cipher(
        "Standard",
        adaptive_password
    )

    decrypted = decrypt_text(
        encrypted_message,
        inverse_map
    )

    return decrypted, "Decryption completed."

In [ ]:
import gradio as gr

def analyze_and_encrypt(
    message,
    password
):

    encrypted, category, confidence, policy = ai_encrypt(
        message,
        password
    )

    return (
        category,
        confidence,
        policy,
        encrypted
    )


with gr.Blocks(
    title="AI-Powered Adaptive Emoji Cipher"
) as demo:

    gr.Markdown(
        """
        # 🔐 AI-Powered Adaptive Emoji Message Encryption

        ### AI-Based Message Classification + Adaptive Emoji Encryption

        The system analyzes the message using Machine Learning,
        classifies its sensitivity, and selects an encryption policy.
        """
    )

    with gr.Tab("🤖 AI Encrypt"):

        gr.Markdown(
            """
            ### Step 1
            Enter your message.

            ### Step 2
            AI analyzes the message.

            ### Step 3
            The system selects an encryption policy.

            ### Step 4
            The message is converted into emoji cipher text.
            """
        )

        message_input = gr.Textbox(
            label="Message",
            lines=6,
            placeholder="Enter your message..."
        )

        password_input = gr.Textbox(
            label="Password (Optional)",
            type="password",
            placeholder="Enter password..."
        )

        analyze_button = gr.Button(
            "🔍 Analyze & Encrypt"
        )

        category_output = gr.Textbox(
            label="🤖 AI Classification"
        )

        confidence_output = gr.Textbox(
            label="📊 AI Confidence"
        )

        policy_output = gr.Textbox(
            label="🔐 Selected Encryption Policy"
        )

        encrypted_output = gr.Textbox(
            label="🛡️ Encrypted Emoji Message",
            lines=8
        )

        analyze_button.click(
            analyze_and_encrypt,
            inputs=[
                message_input,
                password_input
            ],
            outputs=[
                category_output,
                confidence_output,
                policy_output,
                encrypted_output
            ]
        )

    with gr.Tab("🔓 Decrypt"):

        encrypted_input = gr.Textbox(
            label="Encrypted Emoji Message",
            lines=8
        )

        category_input = gr.Dropdown(
            choices=[
                "normal",
                "personal",
                "sensitive",
                "confidential"
            ],
            label="AI Classification"
        )

        decrypt_password = gr.Textbox(
            label="Password",
            type="password"
        )

        decrypt_button = gr.Button(
            "🔓 Decrypt"
        )

        decrypted_output = gr.Textbox(
            label="Original Message",
            lines=6
        )

        decrypt_status = gr.Textbox(
            label="Status"
        )

        decrypt_button.click(
            ai_decrypt,
            inputs=[
                encrypted_input,
                category_input,
                decrypt_password
            ],
            outputs=[
                decrypted_output,
                decrypt_status
            ]
        )

    with gr.Tab("🧠 AI Model Information"):

        gr.Markdown(
            f"""
            ## Machine Learning Model

            **Algorithm:** Logistic Regression

            **Feature Extraction:** TF-IDF

            **Classes:**
            - Normal
            - Personal
            - Sensitive
            - Confidential

            **Training Samples:** {len(df)}

            **Test Accuracy:** {accuracy * 100:.2f}%

            ---

            ### System Workflow

            Message

            ↓

            TF-IDF

            ↓

            Logistic Regression

            ↓

            Message Classification

            ↓

            Adaptive Encryption Policy

            ↓

            Emoji Encryption
            """
        )


demo.launch(
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b4ebcbca5a0890b27d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
